# Phase 1 — tree-based models (handoff.md Sec. 4–5)

* **Models:** LightGBM, XGBoost, CatBoost regressors (spec) + LightGBM **hurdle** (classifier for "price changes"
  + L1 regressor for the size; predicts *no change* unless P(change) > tuned threshold).
* **Target:** `target_pct_change` vs. the causal regular price of t−1; loss = absolute error weighted by price
  (= MAE in TRY). Under MAE the optimal forecast is the conditional *median*, so a model only gains where it can
  find changes that are more likely than not.
* **Tuning:** Optuna + `TimeSeriesSplit` (3 folds over dates) on **B1's training window only**; objective = system
  MAE on target days (spec ∪ data set). The same parameters are reused for B2 and B3.
* **Evaluation:** spec 2.3 system output (model on target days, persistence elsewhere) for the spec and data
  target-day sets; MAE / RMSE / MAPE / DA; improvement vs. both persistence baselines.
* **Model selection on B1 + B2 only.** B3 is the final holdout: reported once, never used for decisions.

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # paths below are relative to the repository root
    os.chdir(Path.cwd().parent)

import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import optuna
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.evaluation import metrics as M
from src.models.classical_ml import phase1 as P
from src.preprocessing.features import FEATURE_GROUPS, feature_columns

Path("logs").mkdir(exist_ok=True)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.FileHandler("logs/phase1.log"), logging.StreamHandler()], force=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)
log = logging.getLogger("phase1")
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

PROCESSED = Path("data/processed")
feat = pd.read_parquet(PROCESSED / "gurmar_features.parquet")
BLOCKS = json.loads((PROCESSED / "gurmar_blocks.json").read_text())["blocks"]
DEV = ["B1", "B2"]
FEATURES = feature_columns()
MODELS = ["lightgbm", "xgboost", "catboost", "lightgbm_hurdle"]
N_TRIALS = {"lightgbm": 40, "xgboost": 30, "catboost": 15, "lightgbm_hurdle": 60}
print(f"{len(feat):,} rows, {len(FEATURES)} features")

1,155,981 rows, 50 features


## 1. Hyperparameter tuning on B1's training window

In [2]:
tune_rows = P.split_rows(feat, BLOCKS[0]["train_start"], BLOCKS[0]["train_end"])
tune_rows = tune_rows[~tune_rows["in_outage"]]
best_params, tuning = {}, []
for name in MODELS:
    start = time.time()
    study = P.tune(name, tune_rows, FEATURES, N_TRIALS[name])
    best_params[name] = study.best_params
    tuning.append({"model": name, "trials": N_TRIALS[name], "best_cv_target_day_MAE": study.best_value,
                   "minutes": round((time.time() - start) / 60, 1), "best_iter": study.best_trial.user_attrs["best_iter"]})

Path("configs").mkdir(exist_ok=True)
Path("configs/phase1_best_params.json").write_text(json.dumps(best_params, indent=2))
pd.DataFrame(tuning)

2026-10-06 14:54:15,032 INFO src.models.classical_ml.phase1: lightgbm: 40 trials in 828s, best fold-mean target-day MAE 0.7058


2026-10-06 15:15:28,878 INFO src.models.classical_ml.phase1: xgboost: 30 trials in 1274s, best fold-mean target-day MAE 0.6903


2026-10-06 15:25:07,230 INFO src.models.classical_ml.phase1: catboost: 15 trials in 578s, best fold-mean target-day MAE 0.8029


2026-10-06 15:50:04,688 INFO src.models.classical_ml.phase1: lightgbm_hurdle: 60 trials in 1497s, best fold-mean target-day MAE 0.5705


,model,trials,best_cv_target_day_MAE,minutes,best_iter
0,lightgbm,40,0.705807,13.8,{'reg': 116}
1,xgboost,30,0.690325,21.2,{'reg': 549}
2,catboost,15,0.802938,9.6,{'reg': 26}
3,lightgbm_hurdle,60,0.570527,25.0,"{'cls': 659, 'reg': 622}"


In [3]:
# persistence on the same CV folds, for reference
from sklearn.model_selection import TimeSeriesSplit
dates = np.sort(tune_rows["date"].unique())
cv_base = [P.target_day_mae(tune_rows[tune_rows["date"].isin(dates[va])], np.zeros(int(tune_rows["date"].isin(dates[va]).sum())))
           for _, va in TimeSeriesSplit(n_splits=3).split(dates)]
print(f"persistence (causal regular) CV target-day MAE: {np.mean(cv_base):.4f}")

persistence (causal regular) CV target-day MAE: 0.8125


## 2. Train per block and evaluate

In [4]:
results, predictions, fitted_models = [], [], {}
for block in BLOCKS:
    test = P.split_rows(feat, block["test_start"], block["test_end"])
    for base_name, base_pred in M.baselines(test).items():
        results.append(M.evaluate(test, base_pred, base_name).assign(block=block["block"]))
    for name in MODELS:
        start = time.time()
        model, test_rows, pct, n_est = P.train_block(name, best_params[name], feat, block, FEATURES)
        price = P.to_price(test_rows, pct)
        results.append(M.evaluate(test_rows, price, name).assign(block=block["block"]))
        predictions.append(test_rows[["segment_id", "date", "is_target_spec", "is_target_data", "base_price", "target_price"]]
                           .assign(block=block["block"], model=name, pred_price=price, pred_pct=pct))
        fitted_models[(block["block"], name)] = model
        log.info("%s %s: trees=%s, %.0fs", block["block"], name, n_est, time.time() - start)

res = M.add_improvement(pd.concat(results, ignore_index=True))
res.to_csv(PROCESSED / "phase1_results.csv", index=False)
pd.concat(predictions).to_parquet(PROCESSED / "phase1_predictions.parquet", index=False)

2026-10-06 15:50:16,148 INFO phase1: B1 lightgbm: trees={'reg': 126}, 11s


2026-10-06 15:50:46,201 INFO phase1: B1 xgboost: trees={'reg': 781}, 30s


2026-10-06 15:51:09,937 INFO phase1: B1 catboost: trees={'reg': 39}, 24s


2026-10-06 15:51:44,631 INFO phase1: B1 lightgbm_hurdle: trees={'cls': 914, 'reg': 549}, 35s


2026-10-06 15:51:54,310 INFO phase1: B2 lightgbm: trees={'reg': 10}, 10s


...

2026-10-06 15:53:14,522 INFO phase1: B2 lightgbm_hurdle: trees={'cls': 293, 'reg': 893}, 41s


2026-10-06 15:54:37,493 INFO phase1: B3 lightgbm: trees={'reg': 406}, 83s


2026-10-06 15:55:27,341 INFO phase1: B3 xgboost: trees={'reg': 662}, 50s


2026-10-06 15:55:55,751 INFO phase1: B3 catboost: trees={'reg': 10}, 28s


2026-10-06 15:58:23,587 INFO phase1: B3 lightgbm_hurdle: trees={'cls': 2327, 'reg': 2327}, 148s


In [5]:
def show(block_ids, subset):
    cols = ["block", "model", "rows", "MAE", "RMSE", "MAPE_%", "DA_%", "impr_vs_causal_regular_%", "impr_vs_best_%",
            "worse_than_best_baseline"]
    t = res[res["block"].isin(block_ids) & (res["subset"] == subset)][cols]
    return t.sort_values(["block", "MAE"]).round(4)

show(DEV, "spec target days")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
25,B1,lightgbm_hurdle,14063,3.4884,22.7672,2.0743,93.0811,6.3599,6.3599,False
10,B1,lightgbm,14063,3.5672,22.8434,2.2206,92.5123,4.2428,4.2428,False
15,B1,xgboost,14063,3.6142,22.4110,2.2276,92.5834,2.9831,2.9831,False
0,B1,persistence_causal_regular,14063,3.7253,23.4183,2.2320,92.3274,0.0000,0.0000,False
20,B1,catboost,14063,3.7255,23.3434,2.2284,92.4340,-0.0044,-0.0044,True
5,B1,persistence_shelf,14063,9.0915,31.1240,4.9068,81.4335,-144.0492,-144.0492,True
55,B2,lightgbm_hurdle,7211,4.0917,33.3872,1.7486,92.4005,18.8907,18.8907,False
40,B2,lightgbm,7211,4.7073,34.1117,2.1019,87.4913,6.6884,6.6884,False
45,B2,xgboost,7211,5.0229,36.8761,2.1974,90.0707,0.4329,0.4329,False
30,B2,persistence_causal_regular,7211,5.0447,37.3574,2.2029,89.9598,0.0000,0.0000,False


In [6]:
show(DEV, "data target days")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
27,B1,lightgbm_hurdle,24010,2.5005,20.0938,1.4687,94.9813,6.5134,6.5134,False
12,B1,lightgbm,24010,2.5698,20.1430,1.6170,94.3149,3.9211,3.9211,False
17,B1,xgboost,24010,2.6049,19.8992,1.6151,94.4690,2.6109,2.6109,False
22,B1,catboost,24010,2.6720,20.5247,1.5996,94.3732,0.1023,0.1023,False
2,B1,persistence_causal_regular,24010,2.6747,20.6645,1.6034,94.3565,0.0000,0.0000,False
7,B1,persistence_shelf,24010,8.0826,29.4689,4.2698,83.2861,-202.1856,-202.1856,True
57,B2,lightgbm_hurdle,14672,3.7909,37.2952,1.4482,94.3225,23.7203,23.7203,False
42,B2,lightgbm,14672,4.5563,38.2929,1.8135,88.2974,8.3190,8.3190,False
47,B2,xgboost,14672,4.9367,41.6002,1.9339,91.9234,0.6646,0.6646,False
32,B2,persistence_causal_regular,14672,4.9697,42.2287,1.9425,91.7530,0.0000,0.0000,False


In [7]:
show(DEV, "global (model on spec days)")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
26,B1,lightgbm_hurdle,139713,1.8109,17.3766,1.1292,96.0927,1.2998,1.2998,False
11,B1,lightgbm,139713,1.8189,17.3867,1.1439,96.0354,0.8671,0.8671,False
16,B1,xgboost,139713,1.8236,17.3300,1.1446,96.0426,0.6096,0.6096,False
1,B1,persistence_causal_regular,139713,1.8348,17.4635,1.1451,96.0168,0.0000,0.0000,False
21,B1,catboost,139713,1.8348,17.4534,1.1447,96.0276,-0.0009,-0.0009,True
6,B1,persistence_shelf,139713,2.3749,18.6354,1.4143,94.9203,-29.4393,-29.4393,True
56,B2,lightgbm_hurdle,62576,3.8659,42.6811,1.2838,95.1898,2.7622,2.7622,False
41,B2,lightgbm,62576,3.9368,42.7471,1.3245,94.6241,0.9780,0.9780,False
46,B2,xgboost,62576,3.9732,43.0108,1.3355,94.9214,0.0633,0.0633,False
31,B2,persistence_causal_regular,62576,3.9757,43.0586,1.3362,94.9086,0.0000,0.0000,False


In [8]:
show(DEV, "all days (model everywhere)")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
29,B1,lightgbm_hurdle,139713,1.7102,17.0394,1.0248,96.6023,6.7928,6.7928,False
14,B1,lightgbm,139713,1.7831,17.1365,1.1666,95.8121,2.8153,2.8153,False
19,B1,xgboost,139713,1.8186,17.1588,1.1673,96.1299,0.8831,0.8831,False
4,B1,persistence_causal_regular,139713,1.8348,17.4635,1.1451,96.0168,0.0000,0.0000,False
24,B1,catboost,139713,1.8599,17.3834,1.1506,95.5645,-1.3663,-1.3663,True
9,B1,persistence_shelf,139713,7.0545,29.1164,3.5686,85.9140,-284.4861,-284.4861,True
59,B2,lightgbm_hurdle,62576,3.3124,39.2959,1.1165,96.0784,16.6831,16.6831,False
44,B2,lightgbm,62576,3.8189,39.8810,1.3039,89.3090,3.9445,3.9445,False
49,B2,xgboost,62576,3.9503,42.3674,1.3332,94.9821,0.6378,0.6378,False
34,B2,persistence_causal_regular,62576,3.9757,43.0586,1.3362,94.9086,0.0000,0.0000,False


### Model selection (B1 + B2)

Best model = highest mean improvement over the better persistence baseline on **spec target days** across B1 and B2.

In [9]:
dev = res[res["block"].isin(DEV) & res["model"].isin(MODELS)]
selection = (
    dev[dev["subset"].isin(["spec target days", "data target days"])]
    .pivot_table(index="model", columns="subset", values="impr_vs_best_%", aggfunc="mean")
    .sort_values("spec target days", ascending=False)
    .round(3)
)
BEST = selection.index[0]
print("Selected model:", BEST)
selection

Selected model: lightgbm_hurdle


subset,data target days,spec target days
model,,
lightgbm_hurdle,15.117,12.625
lightgbm,6.120,5.466
xgboost,1.638,1.708
catboost,-4.182,-3.204


## 3. Ablation (spec 5.3) with the selected model

In [10]:
RUNS = {
    "run1_price_lag_1": (["price_lag_1"], False),
    "run2_lags_cyclic": (feature_columns(["lags", "cyclic"]), False),
    "run3_+usd_try": (feature_columns(["lags", "cyclic", "usd_try"]), False),
    "run4_spec_full (+tuik, rolling)": (feature_columns(["lags", "cyclic", "usd_try", "tuik", "rolling"]), True),
    "run5_all (+oil, sale, unit price, cross-section, web index)": (FEATURES, True),
}
abl = []
for block in BLOCKS:
    for run, (cols, categorical) in RUNS.items():
        _, test_rows, pct, _ = P.train_block(BEST, best_params[BEST], feat, block, cols, categorical)
        ev = M.evaluate(test_rows, P.to_price(test_rows, pct), run).assign(block=block["block"])
        abl.append(ev)
    test = P.split_rows(feat, block["test_start"], block["test_end"])
    for base_name, base_pred in M.baselines(test).items():
        abl.append(M.evaluate(test, base_pred, base_name).assign(block=block["block"]))

ablation = M.add_improvement(pd.concat(abl, ignore_index=True))
ablation.to_csv(PROCESSED / "phase1_ablation.csv", index=False)
(
    ablation[ablation["block"].isin(DEV) & ablation["subset"].isin(["spec target days", "data target days"])
             & ablation["model"].str.startswith("run")]
    .pivot_table(index="model", columns=["block", "subset"], values="impr_vs_best_%")
    .round(2)
)

block                                                            B1                                B2                 
subset                                             data target days spec target days data target days spec target days
model                                                                                                                 
run1_price_lag_1                                               0.00             0.00             0.00             0.00
run2_lags_cyclic                                               0.00             0.00             3.21             1.96
run3_+usd_try                                                  0.00             0.00             1.76             1.16
run4_spec_full (+tuik, rolling)                                4.28             5.00            22.12            17.84
run5_all (+oil, sale, unit price, cross-section...             6.51             6.36            23.72            18.89

## 4. What the selected model uses

In [11]:
model = fitted_models[("B2", BEST)]
if BEST == "lightgbm_hurdle":
    parts = {"classifier": model.models["cls"], "size regressor": model.models["reg"]}
else:
    parts = {"regressor": model.models["reg"]}
imp = {}
for label, m in parts.items():
    if hasattr(m, "booster_"):
        gain = m.booster_.feature_importance(importance_type="gain")
        imp[label] = pd.Series(gain / gain.sum(), index=m.booster_.feature_name())
    elif hasattr(m, "get_feature_importance"):
        v = m.get_feature_importance()
        imp[label] = pd.Series(v / v.sum(), index=m.feature_names_)
    else:
        imp[label] = pd.Series(m.feature_importances_, index=m.get_booster().feature_names)
pd.DataFrame(imp).sort_values(list(imp)[0], ascending=False).head(15).round(3)

,classifier,size regressor
ratio_lag1_mean30,0.155,0.037
shelf_to_base_lag1,0.085,0.106
last_change_size,0.070,0.051
month_sin,0.065,0.001
sale_depth_lag1,0.065,0.076
log_unit_price_lag1,0.058,0.061
rel_unit_price_lag1,0.049,0.055
price_lag_1,0.043,0.056
days_to_month_end,0.029,0.008
days_since_change,0.029,0.037


In [12]:
pred = pd.read_parquet(PROCESSED / "phase1_predictions.parquet")
p = pred[(pred["model"] == BEST) & pred["block"].isin(DEV)].copy()
p["pred_change"] = np.abs(p["pred_price"] / p["base_price"] - 1) > M.TOL
p["true_change"] = np.abs(p["target_price"] / p["base_price"] - 1) > M.TOL
pd.DataFrame({
    "predicted changes": p.groupby("block")["pred_change"].sum(),
    "true changes": p.groupby("block")["true_change"].sum(),
    "precision": p[p["pred_change"]].groupby("block")["true_change"].mean().round(3),
    "recall": p[p["true_change"]].groupby("block")["pred_change"].mean().round(3),
})

,predicted changes,true changes,precision,recall
block,,,,
B1,1656,5565,0.772,0.230
B2,1902,3186,0.719,0.429


## 5. Final holdout B3 (report only — no decisions on these numbers)

In [13]:
show(["B3"], "spec target days")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
60,B3,persistence_causal_regular,6967,0.7253,7.9475,0.3962,98.2202,0.0000,0.0000,False
80,B3,catboost,6967,0.7524,7.8415,0.3990,98.2202,-3.7406,-3.7406,True
85,B3,lightgbm_hurdle,6967,1.0938,18.3019,0.4259,98.5790,-50.8038,-50.8038,True
70,B3,lightgbm,6967,1.3225,17.5868,0.4677,97.5599,-82.3410,-82.3410,True
75,B3,xgboost,6967,1.5354,22.2573,0.4565,98.2489,-111.7032,-111.7032,True
65,B3,persistence_shelf,6967,4.7771,18.5972,2.8987,87.4982,-558.6549,-558.6549,True


In [14]:
show(["B3"], "data target days")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
87,B3,lightgbm_hurdle,3247,0.8593,10.6624,0.4903,98.2445,7.3368,7.3368,False
62,B3,persistence_causal_regular,3247,0.9274,9.2267,0.4857,97.9058,0.0000,0.0000,False
82,B3,catboost,3247,0.9570,9.1302,0.4892,97.8750,-3.1992,-3.1992,True
72,B3,lightgbm,3247,1.4674,16.2631,0.5591,97.2282,-58.2383,-58.2383,True
77,B3,xgboost,3247,1.7801,23.5561,0.5397,97.9674,-91.9481,-91.9481,True
67,B3,persistence_shelf,3247,5.2642,19.6709,3.1794,86.3258,-467.6515,-467.6515,True


In [15]:
show(["B3"], "global (model on spec days)")

,block,model,rows,MAE,RMSE,MAPE_%,DA_%,impr_vs_causal_regular_%,impr_vs_best_%,worse_than_best_baseline
61,B3,persistence_causal_regular,60283,1.1463,23.7219,0.3945,98.3096,0.0000,0.0000,False
81,B3,catboost,60283,1.1494,23.7178,0.3948,98.3096,-0.2735,-0.2735,True
86,B3,lightgbm_hurdle,60283,1.1889,24.3750,0.3979,98.3511,-3.7149,-3.7149,True
71,B3,lightgbm,60283,1.2153,24.3141,0.4028,98.2333,-6.0211,-6.0211,True
76,B3,xgboost,60283,1.2399,24.7524,0.4015,98.3130,-8.1681,-8.1681,True
66,B3,persistence_shelf,60283,1.6146,24.4008,0.6837,97.0705,-40.8508,-40.8508,True
